# Klastering Komentar - Koperasi Desa Merah Putih (KDMP)
Notebook ini melakukan **(1) pembersihan data** dan **(2) klastering tanpa label** untuk menemukan topik/pola pembicaraan alami dalam komentar TikTok, menggunakan **TF-IDF -> TruncatedSVD (LSA) -> K-Means**.

Label `sentimen` **tidak dipakai** saat klastering; hanya dipakai di akhir sebagai pembanding (validasi eksternal).

**Alur:** Audit data -> Cleaning -> Vektorisasi -> Tentukan jumlah klaster -> K-Means -> Interpretasi topik -> Visualisasi -> Bandingkan dengan label -> Algoritma pembanding -> Simpan hasil.

## 1. Setup & Konfigurasi

In [ ]:
# Jalankan sekali jika library belum terpasang (hapus tanda # di baris berikut)
# !pip install -q pandas numpy scikit-learn matplotlib seaborn joblib Sastrawi emoji

In [ ]:
import warnings, re, os, joblib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
# ============== KONFIGURASI ==============
FILE_PATH       = "KDMP_berlabel_perbaikan.csv"   # lokasi file dataset
RANDOM_STATE    = 42
SAMPLE_N        = None    # isi angka (mis. 20000) untuk uji cepat; None = pakai seluruh data
USE_STEMMING    = False   # stemming Sastrawi: bisa menyatukan kata berimbuhan, tapi LAMBAT (~15-30 menit untuk data penuh; hasil di-cache)
CLEAN_CSV_PATH  = "KDMP_bersih.csv"
STEM_CACHE_PATH = "stem_cache.pkl"

## 2. Memuat & Mengaudit Data Mentah
Sebelum dibersihkan, kita periksa dulu masalah apa saja yang ada di dataset. Temuan utama pada data ini:
- Baris **duplikat** (`comment_id` ganda) - ada salinan komentar level 0 dengan kolom `level` kosong.
- Komentar **kosong** dan bernilai `#NAME?` (error spreadsheet).
- `parent_comment_id` rusak karena tersimpan dalam notasi ilmiah (`7.64e+18`), sehingga tidak dipakai sebagai fitur.
- Teks penuh **noise**: tag `[Sticker]`, `\r\n` literal, emoji, huruf berulang, slang, mention, URL.

In [ ]:
# Baca semua kolom sebagai string agar ID besar (19 digit) tidak rusak / berubah jadi notasi ilmiah
df_raw = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=True)
print("Ukuran data mentah :", df_raw.shape)
df_raw.head()

In [ ]:
print("=== Nilai kosong per kolom ===")
print(df_raw.isna().sum(), "\n")

print("=== Distribusi label (mentah) ===")
print(df_raw["sentimen"].value_counts(dropna=False), "\n")

print("=== Distribusi kolom level ===")
print(df_raw["level"].value_counts(dropna=False), "\n")

print("Baris duplikat penuh          :", df_raw.duplicated().sum())
print("comment_id duplikat           :", df_raw["comment_id"].duplicated().sum())
print("Komentar kosong/NaN           :", (df_raw["comment"].isna() | (df_raw["comment"].str.strip() == "")).sum())
print("Komentar bernilai '#NAME?'    :", (df_raw["comment"] == "#NAME?").sum())
print("Komentar berisi tag [Sticker] :", df_raw["comment"].str.contains(r"\[[Ss]ti(c)?ker\]", na=False).sum())
print("Komentar berisi baris baru (\\r\\n):", df_raw["comment"].str.contains(r"[\r\n]", na=False).sum())
print("Contoh parent_comment_id (rusak, notasi ilmiah):", df_raw["parent_comment_id"].dropna().unique()[:3])

## 3. Pembersihan Struktur Data
Label dirapikan, komentar kosong/error dibuang, duplikat dihapus, tipe data diperbaiki.

In [ ]:
log = []   # catatan jumlah baris tiap tahap pembersihan
def catat(tahap, d):
    log.append({"tahap": tahap, "jumlah_baris": len(d)})

df = df_raw.copy()
catat("Data mentah", df)

# 1) Rapikan label
df["sentimen"] = df["sentimen"].str.strip().str.lower()
df = df[df["sentimen"].isin(["positif", "netral", "negatif"])]
catat("Label valid (positif/netral/negatif)", df)

# 2) Buang teks kosong & error spreadsheet ('#NAME?')
df["comment"] = df["comment"].replace({"#NAME?": np.nan, "#N/A": np.nan, "#VALUE!": np.nan})
df["comment"] = df["comment"].str.strip()
df = df[df["comment"].notna() & (df["comment"] != "")]
catat("Komentar tidak kosong", df)

# 3) Hapus duplikat comment_id (file berisi salinan ganda dari komentar level 0)
df = df.drop_duplicates(subset="comment_id", keep="first")
catat("Hapus duplikat comment_id", df)

# 4) Konversi tipe data
df["create_time"] = pd.to_datetime(df["create_time"], errors="coerce")
df = df.dropna(subset=["create_time"])
df["level"] = pd.to_numeric(df["level"], errors="coerce")
# Level: kolom level rusak pada salinan ganda -> tentukan ulang dari parent_comment_id
df["is_reply"] = df["parent_comment_id"].notna().astype(int)
catat("Format tanggal valid", df)

# 5) Kolom ID/parent tidak dipakai sebagai fitur (ID hanya pengenal)
df = df.reset_index(drop=True)

if SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
    catat(f"Sampling {SAMPLE_N} baris", df)

pd.DataFrame(log)

## 4. Pembersihan Teks (Preprocessing NLP)
Tahapan: *lowercase* -> hapus URL/mention/tag stiker/`\r\n` -> normalisasi huruf berulang -> **emoji diubah jadi token teks** (emoji membawa sentimen) -> hapus angka & simbol -> **normalisasi slang** -> hapus *stopword* (kata **negasi** seperti "tidak", "bukan" sengaja dipertahankan) -> *stemming* Sastrawi.

In [ ]:
import re, os, joblib, emoji
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ---- Kamus slang / singkatan bahasa Indonesia -> bentuk baku ----
SLANG = {
    "gk":"tidak","ga":"tidak","gak":"tidak","nggak":"tidak","ngga":"tidak","enggak":"tidak","engga":"tidak",
    "tdk":"tidak","g":"tidak","tak":"tidak","kagak":"tidak","gpp":"tidak apa","gapapa":"tidak apa",
    "yg":"yang","yng":"yang","dgn":"dengan","dg":"dengan","dr":"dari","dri":"dari","utk":"untuk","untk":"untuk",
    "krn":"karena","karna":"karena","krna":"karena","sm":"sama","sama2":"sama","tp":"tapi","tapi":"tapi",
    "jd":"jadi","jdi":"jadi","udh":"sudah","udah":"sudah","sdh":"sudah","dah":"sudah","blm":"belum","belom":"belum",
    "bgt":"banget","bngt":"banget","bgd":"banget","skrg":"sekarang","skrang":"sekarang","sblm":"sebelum",
    "lg":"lagi","lgi":"lagi","lgsg":"langsung","kalo":"kalau","klo":"kalau","kl":"kalau","klw":"kalau",
    "emg":"memang","emang":"memang","mmg":"memang","hrs":"harus","harusny":"harus","bs":"bisa","bsa":"bisa",
    "sy":"saya","aq":"aku","ak":"aku","gw":"aku","gue":"aku","gua":"aku","w":"aku","lu":"kamu","loe":"kamu",
    "km":"kamu","kmu":"kamu","org":"orang","orng":"orang","spt":"seperti","sprti":"seperti","kyk":"seperti",
    "kayak":"seperti","bkn":"bukan","bukannya":"bukan","brp":"berapa","brapa":"berapa",
    "dmn":"dimana","dmna":"dimana","kmn":"kemana","knp":"kenapa","kenapa":"kenapa","gmn":"bagaimana","gimana":"bagaimana",
    "gmna":"bagaimana","bgmn":"bagaimana","emangnya":"memang","pd":"pada","pdhl":"padahal","pdahal":"padahal",
    "hrg":"harga","hrga":"harga","rb":"ribu","rbu":"ribu","jt":"juta","tmn":"teman","tmen":"teman",
    "mksh":"terima kasih","makasih":"terima kasih","thx":"terima kasih","tks":"terima kasih","trims":"terima kasih",
    "mantul":"mantap","mantab":"mantap","mantaap":"mantap","sip":"bagus","oke":"bagus","ok":"bagus","okey":"bagus",
    "bagu":"bagus","top":"bagus","keren":"bagus","joss":"bagus","sukses":"sukses","smoga":"semoga","semoga":"semoga",
    "moga":"semoga","amin":"amin","aamiin":"amin","amiin":"amin","btw":"omong omong","jgn":"jangan","jangn":"jangan",
    "sbg":"sebagai","sbgai":"sebagai","tsb":"tersebut","dll":"dan lain lain","dsb":"dan sebagainya","spy":"supaya",
    "biar":"supaya","cm":"cuma","cuman":"cuma","doang":"saja","aja":"saja","aj":"saja","sj":"saja","saja":"saja",
    "trus":"terus","trs":"terus","terus2":"terus","bnyk":"banyak","banyakk":"banyak","sdikit":"sedikit","skit":"sedikit",
    "lbh":"lebih","lbih":"lebih","mgkn":"mungkin","mungkin":"mungkin","tau":"tahu","tw":"tahu","pke":"pakai","pake":"pakai",
    "bikin":"buat","bkin":"buat","liat":"lihat","lht":"lihat","dpt":"dapat","dapet":"dapat","dpat":"dapat","ttp":"tetap",
    "tetep":"tetap","bru":"baru","baru2":"baru","lama2":"lama","cmn":"cuma","sll":"selalu","slalu":"selalu",
    "pemerintah":"pemerintah","pmrntah":"pemerintah","pemrintah":"pemerintah","pmerintah":"pemerintah",
    "indo":"indonesia","ri":"indonesia","wkwk":"tertawa","wkwkwk":"tertawa","wkwkwkwk":"tertawa","haha":"tertawa",
    "hahaha":"tertawa","hehe":"tertawa","hihi":"tertawa","kwkw":"tertawa","xixi":"tertawa","awokawok":"tertawa",
    "anjir":"kaget","anjay":"kaget","njir":"kaget","astaga":"kaget","astagfirullah":"istighfar",
}

# ---- Stopword: Sastrawi + tambahan; kata negasi DIPERTAHANKAN karena penting untuk sentimen ----
NEGASI = {"tidak","bukan","belum","jangan","tanpa","kurang","tak","nggak","gak","ga"}
STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
STOPWORDS |= {"nya","sih","deh","dong","lah","kah","pun","tuh","nih","ya","yah","kan","kok","lho","loh","mah","nah","wah",
              "mas","bang","bg","kak","kakak","sis","bro","min","pak","bapak","bu","ibu","om","tante","mba","mbak","mbk",
              "eh","oh","ah","hehe","hmm","aja","saja","ada","adalah","sih","tertawa","lho","gitu","gini","begitu","begini",
              "banget","sekali","sangat","juga","lagi","udah","sudah","aku","kamu","saya","kita","kami","mereka","dia","nya",
              "kalau","jadi","buat","apa","sama","semua","dulu","terus","tahu","dapat","biar","gitu","display_"}
STOPWORDS -= NEGASI
STOPWORDS -= {"tidak","bukan","belum","jangan","tanpa","kurang"}

# ---- Kata topik yang muncul di hampir semua komentar (tidak membedakan apa-apa) ----
DOMAIN_STOPWORDS = {"koperasi","koprasi","kopdes","kdmp","kmp","merah","putih","desa","kelurahan"}
STOPWORDS |= DOMAIN_STOPWORDS

In [ ]:
URL_RE      = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE  = re.compile(r"@[\w\.]+")
BRACKET_RE  = re.compile(r"\[[^\]]{1,25}\]")            # [Sticker], [laughwithtears], dll.
SKIN_RE     = re.compile(r"_(medium_light|medium_dark|light|medium|dark)_skin_tone")
REPEAT_RE   = re.compile(r"(.)\1{2,}")                   # 3+ karakter/emoji berulang -> 1

stemmer = StemmerFactory().create_stemmer() if USE_STEMMING else None
stem_cache = joblib.load(STEM_CACHE_PATH) if (USE_STEMMING and os.path.exists(STEM_CACHE_PATH)) else {}

NO_STEM = {"pemerintah", "presiden", "rakyat", "menteri"}   # Sastrawi keliru menstem kata ini (mis. pemerintah -> perintah)

def stem_word(w):
    if w in NO_STEM:
        return w
    if w not in stem_cache:
        stem_cache[w] = stemmer.stem(w)
    return stem_cache[w]

def clean_text(text):
    t = str(text)
    t = re.sub(r"\\r\\n|\\n|\\r", " ", t)                 # \r\n yang tertulis literal
    t = re.sub(r"[\r\n\t]+", " ", t)                      # newline sungguhan
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = BRACKET_RE.sub(" ", t)
    t = t.replace("#", " ")
    t = REPEAT_RE.sub(r"\1", t)                           # 'mantaaaap' -> 'mantap', 😂😂😂 -> 😂
    t = emoji.demojize(t, delimiters=(" emo_", " "))      # emoji -> token teks (tetap membawa sentimen)
    t = t.lower()
    t = SKIN_RE.sub("", t)
    t = re.sub(r"[^a-z_\s]", " ", t)                      # buang angka, tanda baca, simbol
    tokens = t.split()

    out = []
    for w in tokens:
        for x in SLANG.get(w, w).split():
            if x.startswith("emo_"):
                out.append(x)
                continue
            if x in STOPWORDS or len(x) < 2:
                continue
            if USE_STEMMING:
                x = stem_word(x)
            if x in STOPWORDS or len(x) < 2:
                continue
            out.append(x)
    # hilangkan token identik yang berurutan
    out = [w for i, w in enumerate(out) if i == 0 or w != out[i-1]]
    return " ".join(out)

# Uji singkat
for contoh in ["Mantaaaap pak!!! 😂😂😂 [Sticker] https://tiktok.com/x @budi", "Gk masuk akal, harga gas 16rb?? Pemerintah bohong\r\nbanget"]:
    print(contoh, "\n  ->", clean_text(contoh))

In [ ]:
df["text_clean"] = df["comment"].apply(clean_text)
if USE_STEMMING:
    joblib.dump(stem_cache, STEM_CACHE_PATH)      # simpan cache agar run berikutnya cepat

df["n_token"] = df["text_clean"].str.split().str.len()
df[["comment", "text_clean", "sentimen"]].sample(8, random_state=RANDOM_STATE)

## 5. Filter Akhir & Simpan Data Bersih

In [ ]:
# Buang komentar yang kosong setelah dibersihkan (mis. hanya stiker / stopword)
df = df[df["n_token"] > 0]
catat("Teks tidak kosong setelah preprocessing", df)

# Hapus teks bersih yang identik (duplikat / spam). Ini juga mencegah kebocoran data train-test.
# Jika teks identik punya label berbeda -> ambigu, dibuang seluruhnya.
n_label = df.groupby("text_clean")["sentimen"].transform("nunique")
df = df[n_label == 1]
catat("Buang teks ambigu (label bertentangan)", df)
df = df.drop_duplicates(subset="text_clean", keep="first").reset_index(drop=True)
catat("Hapus teks duplikat", df)

df = df[["video_id", "comment_id", "username", "create_time", "level", "is_reply",
         "comment", "text_clean", "n_token", "sentimen"]]
df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Data bersih tersimpan: {CLEAN_CSV_PATH}  ({len(df):,} baris)")
pd.DataFrame(log)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame(log).set_index("tahap")["jumlah_baris"].plot.barh(ax=ax[0], color="#4C72B0")
ax[0].invert_yaxis(); ax[0].set_title("Jumlah baris di setiap tahap pembersihan"); ax[0].set_xlabel("baris")

urutan = ["negatif", "netral", "positif"]
warna  = {"negatif": "#C44E52", "netral": "#8C8C8C", "positif": "#55A868"}
cnt = df["sentimen"].value_counts().reindex(urutan)
ax[1].bar(cnt.index, cnt.values, color=[warna[k] for k in cnt.index])
for i, v in enumerate(cnt.values):
    ax[1].text(i, v, f"{v:,}\n({v/cnt.sum():.1%})", ha="center", va="bottom")
ax[1].set_title("Distribusi label setelah dibersihkan"); ax[1].set_ylim(0, cnt.max()*1.2)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
sns.histplot(df["n_token"].clip(upper=40), bins=40, ax=ax[0], color="#4C72B0")
ax[0].set_title("Panjang komentar (jumlah token, dipotong di 40)")
top = pd.Series(" ".join(df["text_clean"]).split()).value_counts().head(20)[::-1]
top.plot.barh(ax=ax[1], color="#DD8452"); ax[1].set_title("20 kata paling sering")
plt.tight_layout(); plt.show()

## 6. Vektorisasi TF-IDF & Reduksi Dimensi (LSA)
TF-IDF menghasilkan matriks yang sangat *sparse* & berdimensi tinggi. Kita reduksi dengan **TruncatedSVD** ke 100 dimensi lalu dinormalisasi (L2) agar jarak Euclidean setara dengan jarak kosinus - standar untuk klastering teks.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.5, sublinear_tf=True, max_features=30_000)
X_tfidf = tfidf.fit_transform(df["text_clean"])

# Buang baris yang vektornya nol (semua katanya di luar kosakata)
ada_fitur = X_tfidf.getnnz(axis=1) > 0
print(f"Baris dengan vektor kosong dibuang: {(~ada_fitur).sum():,}")
df = df[ada_fitur].reset_index(drop=True)
X_tfidf = X_tfidf[ada_fitur]

N_COMP = 100
svd = TruncatedSVD(n_components=N_COMP, random_state=RANDOM_STATE)
X_svd = Normalizer(copy=False).fit_transform(svd.fit_transform(X_tfidf))
print("TF-IDF :", X_tfidf.shape, "-> LSA :", X_svd.shape)
print(f"Variansi yang dipertahankan: {svd.explained_variance_ratio_.sum():.1%}")

## 7. Menentukan Jumlah Klaster (k)
Diuji k = 2 s.d. 10 dengan tiga metrik: **Inertia** (metode *elbow*), **Silhouette** (makin tinggi makin baik), dan **Davies-Bouldin** (makin rendah makin baik). Silhouette dihitung pada sampel 10.000 titik agar cepat.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

rng = np.random.RandomState(RANDOM_STATE)
idx_sil = rng.choice(len(X_svd), size=min(10_000, len(X_svd)), replace=False)

ks, inertia, sil, dbi = list(range(2, 11)), [], [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=3, random_state=RANDOM_STATE).fit(X_svd)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_svd[idx_sil], km.labels_[idx_sil]))
    dbi.append(davies_bouldin_score(X_svd, km.labels_))
    print(f"k={k:2d} | inertia={km.inertia_:10.1f} | silhouette={sil[-1]:.4f} | davies-bouldin={dbi[-1]:.4f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, v, j, w in zip(ax, [inertia, sil, dbi], ["Inertia (Elbow)", "Silhouette (maks. lebih baik)", "Davies-Bouldin (min. lebih baik)"], ["#4C72B0", "#55A868", "#C44E52"]):
    a.plot(ks, v, marker="o", color=w); a.set_title(j); a.set_xlabel("k")
plt.tight_layout(); plt.show()

In [ ]:
# Pilih k. Silhouette pada data teks cenderung memilih k=2 (pemisahan trivial),
# sehingga k dicari mulai dari K_MIN agar hasilnya lebih informatif. Ubah K_FINAL manual bila perlu.
K_MIN   = 4
K_FINAL = None          # isi angka untuk menetapkan k secara manual
if K_FINAL is None:
    kandidat_k = [(k, s) for k, s in zip(ks, sil) if k >= K_MIN]
    K_FINAL = max(kandidat_k, key=lambda x: x[1])[0]
print("Jumlah klaster yang dipakai (k) =", K_FINAL)

## 8. K-Means Final

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, n_init=10, random_state=RANDOM_STATE).fit(X_svd)
df["cluster"] = kmeans.labels_

print(f"Silhouette (sampel) : {silhouette_score(X_svd[idx_sil], kmeans.labels_[idx_sil]):.4f}")
print(f"Davies-Bouldin      : {davies_bouldin_score(X_svd, kmeans.labels_):.4f}\n")

ukuran = df["cluster"].value_counts().sort_index()
ax = ukuran.plot.bar(figsize=(7, 3.5), color="#4C72B0", rot=0)
for i, v in enumerate(ukuran.values):
    ax.text(i, v, f"{v:,}\n{v/len(df):.1%}", ha="center", va="bottom")
ax.set_title("Jumlah komentar per klaster"); ax.set_xlabel("Klaster"); ax.set_ylim(0, ukuran.max()*1.2)
plt.tight_layout(); plt.show()

## 9. Interpretasi Klaster: Kata Kunci & Contoh Komentar

In [ ]:
terms = np.array(tfidf.get_feature_names_out())
top_terms = {}
for c in range(K_FINAL):
    mean_vec = np.asarray(X_tfidf[(df["cluster"] == c).values].mean(axis=0)).ravel()
    top_terms[c] = terms[mean_vec.argsort()[::-1][:12]]
    print(f"Klaster {c} ({(df['cluster']==c).sum():,} komentar): {', '.join(top_terms[c])}")

In [ ]:
for c in range(K_FINAL):
    print(f"\n=== Klaster {c} - {', '.join(top_terms[c][:5])} ===")
    contoh = df[df["cluster"] == c].sample(5, random_state=RANDOM_STATE)["comment"]
    for t in contoh:
        print("  -", t.replace("\\r\\n", " ").replace("\n", " ")[:160])

In [ ]:
# Beri nama klaster setelah membaca kata kunci & contoh di atas.
# Contoh: {0: "Keluhan harga & stok", 1: "Pujian & dukungan program", ...}
cluster_names = {}
df["cluster_name"] = df["cluster"].map(lambda c: cluster_names.get(c, f"Klaster {c}: " + ", ".join(top_terms[c][:3])))
df["cluster_name"].value_counts()

## 10. Visualisasi Klaster (2D)
PCA (linier, cepat) dan t-SNE (menonjolkan struktur lokal) pada sampel data.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

idx_vis = rng.choice(len(X_svd), size=min(8000, len(X_svd)), replace=False)
pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_svd[idx_vis])

idx_tsne = idx_vis[:4000]
tsne2 = TSNE(n_components=2, perplexity=40, init="pca", random_state=RANDOM_STATE).fit_transform(X_svd[idx_tsne])

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(x=pca2[:, 0], y=pca2[:, 1], hue=df["cluster"].values[idx_vis], palette="tab10", s=8, alpha=.6, ax=ax[0], legend="full")
ax[0].set_title("PCA (2 komponen)")
sns.scatterplot(x=tsne2[:, 0], y=tsne2[:, 1], hue=df["cluster"].values[idx_tsne], palette="tab10", s=8, alpha=.7, ax=ax[1], legend="full")
ax[1].set_title("t-SNE")
plt.tight_layout(); plt.show()

## 11. Validasi Eksternal: Klaster vs Label Sentimen
Karena dataset punya label, kita cek apakah klaster berkorelasi dengan sentimen. Nilai ARI/NMI yang **rendah tidak berarti klaster jelek** - klaster teks biasanya membentuk **topik** (harga, stok, korupsi, dll.), bukan sentimen; sebuah topik bisa berisi opini positif maupun negatif.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_completeness_v_measure

h, c, v = homogeneity_completeness_v_measure(df["sentimen"], df["cluster"])
print(f"Adjusted Rand Index : {adjusted_rand_score(df['sentimen'], df['cluster']):.4f}")
print(f"Normalized MI       : {normalized_mutual_info_score(df['sentimen'], df['cluster']):.4f}")
print(f"Homogeneity / Completeness / V-measure : {h:.4f} / {c:.4f} / {v:.4f}\n")

tab = pd.crosstab(df["cluster"], df["sentimen"])
tab_pct = pd.crosstab(df["cluster"], df["sentimen"], normalize="index")[["negatif", "netral", "positif"]]
display(tab)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(tab_pct, annot=True, fmt=".1%", cmap="Blues", ax=ax[0]); ax[0].set_title("Komposisi sentimen di tiap klaster")
tab_pct.plot.bar(stacked=True, ax=ax[1], color=["#C44E52", "#8C8C8C", "#55A868"], rot=0)
ax[1].set_title("Proporsi sentimen per klaster"); ax[1].set_ylabel("proporsi")
plt.tight_layout(); plt.show()

## 12. Algoritma Pembanding: Agglomerative (Ward)
Dijalankan pada sampel (algoritma hierarkis mahal untuk data besar) dan dibandingkan dengan K-Means pada sampel yang sama.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

idx_agg = rng.choice(len(X_svd), size=min(6000, len(X_svd)), replace=False)
Xs = X_svd[idx_agg]

agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward").fit(Xs)
km_s = kmeans.predict(Xs)

banding = pd.DataFrame({
    "Algoritma": ["K-Means", "Agglomerative (Ward)"],
    "Silhouette": [silhouette_score(Xs, km_s), silhouette_score(Xs, agg.labels_)],
    "Davies-Bouldin": [davies_bouldin_score(Xs, km_s), davies_bouldin_score(Xs, agg.labels_)],
    "NMI vs label sentimen": [normalized_mutual_info_score(df["sentimen"].values[idx_agg], km_s),
                              normalized_mutual_info_score(df["sentimen"].values[idx_agg], agg.labels_)],
}).round(4)
print(f"Kesesuaian K-Means vs Agglomerative (ARI): {adjusted_rand_score(km_s, agg.labels_):.4f}")
banding

## 13. Simpan Hasil

In [ ]:
hasil = df[["video_id", "comment_id", "comment", "text_clean", "sentimen", "cluster", "cluster_name"]]
hasil.to_csv("hasil_klaster_KDMP.csv", index=False, encoding="utf-8-sig")
joblib.dump({"tfidf": tfidf, "svd": svd, "kmeans": kmeans}, "model_klaster_kdmp.joblib")
print("Tersimpan: hasil_klaster_KDMP.csv & model_klaster_kdmp.joblib")

def prediksi_klaster(komentar):
    """Tentukan klaster untuk komentar baru."""
    if isinstance(komentar, str):
        komentar = [komentar]
    m = joblib.load("model_klaster_kdmp.joblib")
    v = Normalizer().fit_transform(m["svd"].transform(m["tfidf"].transform([clean_text(k) for k in komentar])))
    return pd.DataFrame({"komentar": komentar, "klaster": m["kmeans"].predict(v)})

prediksi_klaster(["Harga barang di koperasi lebih murah dari warung", "Ini pasti ujung-ujungnya korupsi"])

## 14. Kesimpulan
- Setelah pembersihan, komentar direpresentasikan dengan TF-IDF + LSA dan dikelompokkan dengan K-Means ke dalam beberapa klaster topik (lihat kata kunci & contoh komentar tiap klaster).
- Nilai Silhouette pada data teks biasanya rendah (< 0.1) karena teks pendek & informal; yang lebih penting adalah **keterbacaan topik** tiap klaster.
- Klaster lebih mencerminkan **topik** daripada **sentimen**, terlihat dari komposisi label yang mirip di antara klaster.
- Ide pengembangan: embedding IndoBERT/Sentence-BERT + HDBSCAN atau BERTopic, serta membuang klaster "komentar terlalu pendek/emoji saja" sebagai klaster tersendiri.